# 00 - Data preparation

This notebook does one thing: build the analysis panel from raw MLRan and write
`results/panel.parquet`. All logic lives in the `sift` package; no function or
class is defined here.

The output carries three things reproducibility needs: the provenance table, that
is how many rows survive each filtering step, the content fingerprint of the
panel, and the fingerprints of the three source files together with the MLRan
commit.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from sift import cache
from sift.config import PanelSpec
from sift.data import build_panel, load_mlran

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

## 1. Load the raw data

`load_mlran` joins the two RFE feature files to the metadata. Nothing is filtered
at this step.


In [ ]:
raw = load_mlran()
print("raw shape:", raw.shape)
raw.head(3)

## 2. Build the panel and read the provenance table

The preprocessing steps (`drop_empty`, `dedup_exact`, `min_class_20`) are fixed
and applied identically to all 16 configurations. They are **not** lattice
controls.


In [ ]:
spec = PanelSpec()
panel, provenance = build_panel(raw, spec)

provenance_table = pd.DataFrame(
    {"step": list(provenance), "rows_remaining": list(provenance.values())}
)
provenance_table["rows_dropped"] = (
    provenance_table["rows_remaining"].shift(1) - provenance_table["rows_remaining"]
)
print(provenance_table.to_string(index=False))

In [ ]:
print("panel shape:", panel.shape)
print("n families:", panel["ransomware_family"].nunique())
print("year range:", panel["first_submission_date_year"].min(), "-", panel["first_submission_date_year"].max())
panel["first_submission_date_year"].value_counts().sort_index()

## 3. Fingerprints

The panel fingerprint must agree across machines. If it does not, the lattice
results are not comparable, and the cause has to be traced back to the MLRan
commit or to the sha256 of the three source files below.


In [ ]:
print("panel fingerprint:", cache.frame_fingerprint(panel))
print("code version:", cache.code_version()[:16])

dataset = cache.dataset_fingerprint()
print("mlran commit:", dataset["mlran_commit"])
for name, digest in dataset["files"].items():
    print("  ", name, digest[:16])

## 4. Write the panel

zstd level 3, the same as the two result tables, so every artefact under
`results/` follows one convention.


In [ ]:
panel_path = cache.RESULTS_DIR / "panel.parquet"
panel_path.parent.mkdir(parents=True, exist_ok=True)

pq.write_table(
    pa.Table.from_pandas(panel, preserve_index=False),
    panel_path,
    compression="zstd",
    compression_level=3,
)
print("wrote", panel_path, "=", round(panel_path.stat().st_size / 1e6, 2), "MB")